# 시계열 다루기

> 파이썬 14강 · 시계열과 예측

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [시계열 다루기](https://mioon1402.github.io/timeseriesdata/python/p14-timeseries.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 시계열 인덱스 만들기

**14-1. asfreq 로 빈틈 확인**

In [ ]:
import pandas as pd

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])
ts = df.set_index("date").asfreq("D")     # D = 하루 간격

print("빈도:", ts.index.freq)
print("행 수:", len(ts))
print()
print("매출 결측:", ts["sales"].isna().sum(), "일")
print("빠진 날짜:", ts.index.difference(df["date"]).tolist())

## 2. resample — 시간 단위 바꾸기

**14-2. 여러 단위로 묶어보기**

In [ ]:
s = ts["sales"]

행 = []
for 코드, 이름 in [("W", "주"), ("ME", "월"), ("QE", "분기")]:
    묶음 = s.resample(코드).mean()
    행.append({"단위": 이름, "구간 수": len(묶음), "첫 구간 평균": round(묶음.iloc[0])})

pd.DataFrame(행)

**14-3. 합계와 평균은 다릅니다**

In [ ]:
월합계 = s.resample("ME").sum()
월평균 = s.resample("ME").mean()
일수   = s.resample("ME").size()

표 = pd.DataFrame({
    "일수": 일수,
    "월 총매출(만원)": 월합계 / 10000,
    "일평균(만원)": 월평균 / 10000,
}).round(1)
print(표.head(6).to_string())

## 3. rolling — 이동평균

**14-4. 이동평균 그려보기**

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(s.index, s / 10000, linewidth=0.5, color="#cbd5e1", label="원본(일별)")
ax.plot(s.index, s.rolling(7).mean() / 10000, linewidth=1.5,
        color="#2563eb", label="7일 이동평균")
ax.plot(s.index, s.rolling(30).mean() / 10000, linewidth=2,
        color="#dc2626", label="30일 이동평균")

ax.set_ylabel("매출(만원)")
ax.set_title("이동평균으로 추세 보기", fontweight="bold", loc="left")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

**14-5. 창 크기에 따른 매끄러움**

In [ ]:
행 = [{"창 크기": "원본", "표준편차": round(s.std())}]
for w in [7, 14, 30, 90]:
    행.append({"창 크기": f"{w}일 이동평균", "표준편차": round(s.rolling(w).mean().std())})

pd.DataFrame(행)

**14-6. center=True — 시차 없애기**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
최근 = s.loc["2025-06":"2025-09"] / 10000

ax.plot(최근.index, 최근.values, lw=0.6, color="#cbd5e1", label="원본")
ax.plot(최근.index, 최근.rolling(14).mean(), lw=2,
        color="#dc2626", label="14일 이동평균 (뒤로 밀림)")
ax.plot(최근.index, 최근.rolling(14, center=True).mean(), lw=2,
        color="#0d9488", label="14일 중심이동평균")

ax.legend(frameon=False, fontsize=9)
ax.set_ylabel("매출(만원)")
ax.set_title("center=True 를 쓰면 시차가 사라진다", fontweight="bold", loc="left")
plt.tight_layout(); plt.show()

## 4. shift와 diff

**14-7. 과거 값 가져오기**

In [ ]:
표 = pd.DataFrame({
    "오늘":     s,
    "어제":     s.shift(1),
    "일주일전": s.shift(7),
    "전일대비": s.diff(1),
    "전주대비": s.diff(7),
})
print((표.head(9) / 10000).round(1).to_string())

## 5. 변화율

**14-8. 전월 대비 / 전년 동월 대비**

In [ ]:
월 = s.resample("ME").mean()

성장 = pd.DataFrame({
    "일평균매출(만원)": 월 / 10000,
    "전월대비(%)":      월.pct_change() * 100,
    "전년동월대비(%)":  월.pct_change(12) * 100,
}).round(1)

print(성장.tail(8).to_string())

**14-9. 두 관점 비교해서 그리기**

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.6))
yoy = (월.pct_change(12) * 100).dropna()
mom = (월.pct_change() * 100).dropna()

ax.bar(mom.index, mom.values, width=20, color="#cbd5e1", label="전월 대비")
ax.plot(yoy.index, yoy.values, color="#2563eb", marker="o",
        linewidth=2, label="전년 동월 대비")
ax.axhline(0, color="black", linewidth=0.8)

ax.set_ylabel("증감률(%)")
ax.set_title("전월 대비는 출렁이지만, 전년 대비는 꾸준한 성장을 보여준다",
             fontweight="bold", loc="left", fontsize=12)
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.tight_layout(); plt.show()

## 6. 시계열에서 절대 하면 안 되는 것

**14-10. 누수 없는 결측 처리**

In [ ]:
구멍 = s.copy()

# ✗ 전체 평균으로 채우기 — 미래 정보가 들어간다
나쁨 = 구멍.fillna(구멍.mean())

# ○ 직전 값으로 채우기 — 그 시점에 알 수 있던 정보만
좋음 = 구멍.ffill()

# ○ 확장 평균 — '그때까지의' 평균만 사용
확장 = 구멍.fillna(구멍.expanding().mean())

print("결측:", 구멍.isna().sum(), "건")
print(f"전체평균 대체  {나쁨.mean():>10,.0f}   ← 미래 정보 사용 (예측엔 반칙)")
print(f"직전값 대체    {좋음.mean():>10,.0f}   ← 안전")
print(f"확장평균 대체  {확장.mean():>10,.0f}   ← 안전")

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 방문객(visitors)의 28일 이동평균을 그려보세요.
#        7일 이동평균과 무엇이 다른가요?


# 문제 2. 주별(W) 매출 합계를 구하고, 전주 대비 변화율을 계산해보세요.


# 문제 3. 각 요일별로 '전주 같은 요일 대비' 변화율을 구해보세요.
#        힌트: pct_change(7)

**모범 답안**

In [ ]:
v = ts["visitors"]

# 문제 1
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(v.index, v, lw=0.4, color="#e2e8f0", label="원본")
ax.plot(v.index, v.rolling(7).mean(), lw=1.2, label="7일")
ax.plot(v.index, v.rolling(28).mean(), lw=2, label="28일")
ax.legend(frameon=False); ax.set_ylabel("방문객(명)")
ax.set_title("창이 클수록 매끄럽지만 반응이 느리다", fontweight="bold", loc="left")
plt.tight_layout(); plt.show()

# 문제 2
주 = s.resample("W").sum()
주표 = pd.DataFrame({"주매출(만원)": 주/10000, "전주대비(%)": 주.pct_change()*100})
print(주표.round(1).tail(6).to_string())

# 문제 3
전주대비 = s.pct_change(7) * 100
print()
print("요일별 '전주 같은 요일 대비' 평균 변화율(%)")
print(전주대비.groupby(전주대비.index.dayofweek).mean().round(2).to_string())

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)